In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install timm grad-cam -q

import torch
print(torch.cuda.get_device_name(0))

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 114.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
NVIDIA A100-SXM4-80GB


In [ ]:
"""
MURA Shortcut Learning Project
Full pipeline: dataloader → train → evaluate (clean + masked) → GradCAM
"""

import os, glob, numpy as np, pandas as pd
from PIL import Image
from sklearn.metrics import roc_auc_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import timm

# Config
DATA_ROOT  = "/content/drive/MyDrive/Ml_Project class/MURA_FOR_OUR_PROJ"
BODY_PARTS = ["XR_WRIST", "XR_ELBOW", "XR_SHOULDER"]
BATCH_SIZE = 32
EPOCHS     = 20
SEEDS      = [42, 0, 1, 7, 21]
LR         = 1e-4
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR   = "/content/drive/MyDrive/Ml_Project class/mura_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Using device: {DEVICE}")


# DATASET
class MURADataset(Dataset):
    def __init__(self, root, split, body_parts, transform=None, mask_artifacts=False):
        self.transform = transform
        self.mask_artifacts = mask_artifacts
        self.samples = []
        for bp in body_parts:
            bp_path = os.path.join(root, split, bp)
            if not os.path.exists(bp_path):
                print(f"Warning: {bp_path} not found, skipping.")
                continue
            for study_dir in glob.glob(os.path.join(bp_path, "*", "*")):
                label = 1 if "positive" in study_dir.lower() else 0
                for img_path in glob.glob(os.path.join(study_dir, "*.png")):
                    self.samples.append((img_path, label))
        print(f"[{split}] Loaded {len(self.samples)} images across {body_parts}")

    def __len__(self):
        return len(self.samples)

    def _apply_mask(self, img_tensor):
        _, H, W = img_tensor.shape
        corner = int(0.15 * min(H, W))
        border = 10
        masked = img_tensor.clone()
        masked[:, :corner, :corner] = 0
        masked[:, :corner, -corner:] = 0
        masked[:, -corner:, :corner] = 0
        masked[:, -corner:, -corner:] = 0
        masked[:, :border, :] = 0
        masked[:, -border:, :] = 0
        masked[:, :, :border] = 0
        masked[:, :, -border:] = 0
        return masked

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        if self.mask_artifacts:
            img = self._apply_mask(img)
        return img, label, img_path


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


# MODELS
def get_model(name):
    if name == "resnet50":
        m = models.resnet50(weights="IMAGENET1K_V1")
        m.fc = nn.Linear(m.fc.in_features, 1)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights="IMAGENET1K_V1")
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    elif name == "densenet169":
        m = models.densenet169(weights="IMAGENET1K_V1")
        m.classifier = nn.Linear(m.classifier.in_features, 1)
    elif name == "vit":
        m = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=1)
    else:
        raise ValueError(f"Unknown model: {name}")
    return m.to(DEVICE)


# TRAINING
def train_model(model, train_loader, val_loader, model_name, body_part, seed):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.BCEWithLogitsLoss()
    best_auc = 0
    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.float().unsqueeze(1).to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()
        auc = evaluate(model, val_loader)
        print(f"[{model_name} | {body_part} | seed={seed}] Epoch {epoch+1}/{EPOCHS} "
              f"Loss: {running_loss/len(train_loader):.4f}  Val AUC: {auc:.4f}")
        if auc > best_auc:
            best_auc = auc
            ckpt_path = os.path.join(SAVE_DIR, f"{model_name}_{body_part}_seed{seed}_best.pt")
            torch.save(model.state_dict(), ckpt_path)
    print(f"Best AUC for {model_name} on {body_part} seed={seed}: {best_auc:.4f}")
    return best_auc


# EVALUATION
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs = imgs.to(DEVICE)
            logits = model(imgs).squeeze(1).cpu()
            preds = torch.sigmoid(logits).numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    if len(set(all_labels)) < 2:
        return 0.5
    return roc_auc_score(all_labels, all_preds)


# ENSEMBLE EVALUATION
def evaluate_ensemble(models_dict, loader, weights):
    for m in models_dict.values():
        m.eval()
    all_labels, all_preds = [], []
    with torch.no_grad():
        for imgs, labels, _ in loader:
            imgs = imgs.to(DEVICE)
            weighted_sum = torch.zeros(imgs.size(0))
            total_weight = sum(weights.values())
            for name, model in models_dict.items():
                logits = model(imgs).squeeze(1).cpu()
                probs = torch.sigmoid(logits)
                weighted_sum += (weights[name] / total_weight) * probs
            all_preds.extend(weighted_sum.numpy())
            all_labels.extend(labels.numpy())
    return roc_auc_score(all_labels, all_preds)


# GRADCAM
def generate_gradcam(model, model_name, loader, save_dir, n_images=5):
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    import cv2
    os.makedirs(save_dir, exist_ok=True)
    if model_name == "resnet50":
        target_layers = [model.layer4[-1]]
    elif model_name == "efficientnet_b0":
        target_layers = [model.features[-1]]
    elif model_name == "densenet169":
        target_layers = [model.features.denseblock4]
    elif model_name == "vit":
        target_layers = [model.blocks[-1].norm1]
    else:
        return
    def reshape_transform(tensor, height=14, width=14):
        result = tensor[:, 1:, :].reshape(tensor.size(0), height, width, tensor.size(2))
        return result.transpose(2, 3).transpose(1, 2)
    if model_name == "vit":
        cam = GradCAM(model=model, target_layers=target_layers, reshape_transform=reshape_transform)
    else:
        cam = GradCAM(model=model, target_layers=target_layers)
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    count = 0
    for imgs, labels, paths in loader:
        for i in range(imgs.size(0)):
            if count >= n_images:
                break
            img_tensor = imgs[i].unsqueeze(0).to(DEVICE)
            grayscale_cam = cam(input_tensor=img_tensor)[0]
            img_np = imgs[i].permute(1,2,0).numpy()
            img_np = (img_np * std + mean).clip(0, 1).astype(np.float32)
            visualization = show_cam_on_image(img_np, grayscale_cam, use_rgb=True)
            label_str = "abnormal" if labels[i].item() == 1 else "normal"
            out_path = os.path.join(save_dir, f"{model_name}_{label_str}_{count}.png")
            cv2.imwrite(out_path, cv2.cvtColor(visualization, cv2.COLOR_RGB2BGR))
            count += 1
        if count >= n_images:
            break
    print(f"Saved {count} GradCAM images to {save_dir}")


# MAIN
results = {}

ensemble_configs = {
    "resnet50+efficientnet_b0":              ["resnet50", "efficientnet_b0"],
    "resnet50+densenet169":                  ["resnet50", "densenet169"],
    "efficientnet_b0+densenet169":           ["efficientnet_b0", "densenet169"],
    "resnet50+efficientnet_b0+densenet169":  ["resnet50", "efficientnet_b0", "densenet169"],
    "resnet50+vit":                          ["resnet50", "vit"],
    "efficientnet_b0+vit":                   ["efficientnet_b0", "vit"],
    "resnet50+efficientnet_b0+vit":          ["resnet50", "efficientnet_b0", "vit"],
}

for bp in BODY_PARTS:
    print(f"\n{'='*60}\nBody part: {bp}\n{'='*60}")
    results[bp] = {}

    train_ds      = MURADataset(DATA_ROOT, "train", [bp], transform=train_transform)
    val_ds        = MURADataset(DATA_ROOT, "valid", [bp], transform=val_transform)
    val_masked_ds = MURADataset(DATA_ROOT, "valid", [bp], transform=val_transform, mask_artifacts=True)

    train_loader      = DataLoader(train_ds,      batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    val_loader        = DataLoader(val_ds,        batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    val_masked_loader = DataLoader(val_masked_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    val_aucs       = {}
    trained_models = {}

    for model_name in ["resnet50", "efficientnet_b0", "densenet169", "vit"]:
        seed_clean_aucs  = []
        seed_masked_aucs = []

        for seed in SEEDS:
            torch.manual_seed(seed)
            np.random.seed(seed)

            print(f"\n--- Training {model_name} | seed={seed} ---")
            model = get_model(model_name)
            train_model(model, train_loader, val_loader, model_name, bp, seed)

            ckpt = os.path.join(SAVE_DIR, f"{model_name}_{bp}_seed{seed}_best.pt")
            model.load_state_dict(torch.load(ckpt))

            clean_auc  = evaluate(model, val_loader)
            masked_auc = evaluate(model, val_masked_loader)
            seed_clean_aucs.append(clean_auc)
            seed_masked_aucs.append(masked_auc)

        mean_clean  = round(float(np.mean(seed_clean_aucs)),  4)
        std_clean   = round(float(np.std(seed_clean_aucs)),   4)
        mean_masked = round(float(np.mean(seed_masked_aucs)), 4)
        std_masked  = round(float(np.std(seed_masked_aucs)),  4)
        delta       = round(mean_clean - mean_masked, 4)

        results[bp][model_name] = {
            "clean_auc":  mean_clean,
            "clean_std":  std_clean,
            "masked_auc": mean_masked,
            "masked_std": std_masked,
            "delta_auc":  delta,
        }
        val_aucs[model_name]       = mean_clean
        trained_models[model_name] = model  # last seed's model for GradCAM

        print(f"{model_name} | Clean: {mean_clean:.4f}±{std_clean:.4f} | "
              f"Masked: {mean_masked:.4f}±{std_masked:.4f} | ΔAUC: {delta:.4f}")

        gradcam_dir = f"/content/drive/MyDrive/Ml_Project class/gradcam/{bp}/{model_name}"
        generate_gradcam(model, model_name, val_loader, gradcam_dir, n_images=5)

    for ens_name, members in ensemble_configs.items():
        ens_models  = {m: trained_models[m] for m in members}
        ens_weights = {m: val_aucs[m] for m in members}
        clean_auc  = evaluate_ensemble(ens_models, val_loader,        ens_weights)
        masked_auc = evaluate_ensemble(ens_models, val_masked_loader, ens_weights)
        delta_auc  = round(clean_auc - masked_auc, 4)
        results[bp][ens_name] = {
            "clean_auc":  round(clean_auc,  4),
            "clean_std":  None,
            "masked_auc": round(masked_auc, 4),
            "masked_std": None,
            "delta_auc":  delta_auc,
        }
        print(f"{ens_name} | Clean: {clean_auc:.4f} | Masked: {masked_auc:.4f} | ΔAUC: {delta_auc:.4f}")

# SAVE RESULTS
print("\n\n===== FINAL RESULTS =====")
print(f"{'Model':<40} {'Part':<12} {'Clean AUC':>10} {'±':>4} {'Masked AUC':>11} {'±':>4} {'ΔAUC':>7}")
print("-" * 92)
for bp, models_res in results.items():
    for model_name, metrics in models_res.items():
        std_c = f"{metrics['clean_std']:.4f}" if metrics['clean_std'] is not None else "  -  "
        std_m = f"{metrics['masked_std']:.4f}" if metrics['masked_std'] is not None else "  -  "
        print(f"{model_name:<40} {bp:<12} {metrics['clean_auc']:>10.4f} {std_c:>6} "
              f"{metrics['masked_auc']:>11.4f} {std_m:>6} {metrics['delta_auc']:>7.4f}")

rows = []
for bp, models_res in results.items():
    for model_name, metrics in models_res.items():
        rows.append({"body_part": bp, "model": model_name, **metrics})
df = pd.DataFrame(rows)
csv_path = "/content/drive/MyDrive/Ml_Project class/mura_results.csv"
df.to_csv(csv_path, index=False)
print(f"\nResults saved to {csv_path}")

Using device: cuda

Body part: XR_WRIST
[train] Loaded 9752 images across ['XR_WRIST']
[valid] Loaded 659 images across ['XR_WRIST']
[valid] Loaded 659 images across ['XR_WRIST']

--- Training resnet50 | seed=42 ---
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 150MB/s]


[resnet50 | XR_WRIST | seed=42] Epoch 1/20 Loss: 0.4497  Val AUC: 0.8803
[resnet50 | XR_WRIST | seed=42] Epoch 2/20 Loss: 0.3793  Val AUC: 0.8929
[resnet50 | XR_WRIST | seed=42] Epoch 3/20 Loss: 0.3576  Val AUC: 0.9025
[resnet50 | XR_WRIST | seed=42] Epoch 4/20 Loss: 0.3286  Val AUC: 0.8843
[resnet50 | XR_WRIST | seed=42] Epoch 5/20 Loss: 0.3124  Val AUC: 0.8858
[resnet50 | XR_WRIST | seed=42] Epoch 6/20 Loss: 0.2837  Val AUC: 0.8881
[resnet50 | XR_WRIST | seed=42] Epoch 7/20 Loss: 0.2609  Val AUC: 0.8649
[resnet50 | XR_WRIST | seed=42] Epoch 8/20 Loss: 0.2360  Val AUC: 0.8769
[resnet50 | XR_WRIST | seed=42] Epoch 9/20 Loss: 0.2083  Val AUC: 0.8830
[resnet50 | XR_WRIST | seed=42] Epoch 10/20 Loss: 0.1848  Val AUC: 0.8865
[resnet50 | XR_WRIST | seed=42] Epoch 11/20 Loss: 0.1483  Val AUC: 0.8751
[resnet50 | XR_WRIST | seed=42] Epoch 12/20 Loss: 0.1152  Val AUC: 0.8766
[resnet50 | XR_WRIST | seed=42] Epoch 13/20 Loss: 0.1015  Val AUC: 0.8900
[resnet50 | XR_WRIST | seed=42] Epoch 14/20 Los

100%|██████████| 20.5M/20.5M [00:00<00:00, 178MB/s]


[efficientnet_b0 | XR_WRIST | seed=42] Epoch 1/20 Loss: 0.4762  Val AUC: 0.8736
[efficientnet_b0 | XR_WRIST | seed=42] Epoch 2/20 Loss: 0.3776  Val AUC: 0.8922
[efficientnet_b0 | XR_WRIST | seed=42] Epoch 3/20 Loss: 0.3482  Val AUC: 0.9188
[efficientnet_b0 | XR_WRIST | seed=42] Epoch 4/20 Loss: 0.3178  Val AUC: 0.9037
[efficientnet_b0 | XR_WRIST | seed=42] Epoch 5/20 Loss: 0.2985  Val AUC: 0.8943
[efficientnet_b0 | XR_WRIST | seed=42] Epoch 6/20 Loss: 0.2736  Val AUC: 0.8987
[efficientnet_b0 | XR_WRIST | seed=42] Epoch 7/20 Loss: 0.2469  Val AUC: 0.8986
[efficientnet_b0 | XR_WRIST | seed=42] Epoch 8/20 Loss: 0.2282  Val AUC: 0.9044
[efficientnet_b0 | XR_WRIST | seed=42] Epoch 9/20 Loss: 0.2050  Val AUC: 0.8937
[efficientnet_b0 | XR_WRIST | seed=42] Epoch 10/20 Loss: 0.1877  Val AUC: 0.9031
[efficientnet_b0 | XR_WRIST | seed=42] Epoch 11/20 Loss: 0.1623  Val AUC: 0.8997
[efficientnet_b0 | XR_WRIST | seed=42] Epoch 12/20 Loss: 0.1507  Val AUC: 0.8924
[efficientnet_b0 | XR_WRIST | seed=42

100%|██████████| 54.7M/54.7M [00:00<00:00, 234MB/s]


[densenet169 | XR_WRIST | seed=42] Epoch 1/20 Loss: 0.4419  Val AUC: 0.8882
[densenet169 | XR_WRIST | seed=42] Epoch 2/20 Loss: 0.3672  Val AUC: 0.9064
[densenet169 | XR_WRIST | seed=42] Epoch 3/20 Loss: 0.3420  Val AUC: 0.8920
[densenet169 | XR_WRIST | seed=42] Epoch 4/20 Loss: 0.3096  Val AUC: 0.8983
[densenet169 | XR_WRIST | seed=42] Epoch 5/20 Loss: 0.2812  Val AUC: 0.8896
[densenet169 | XR_WRIST | seed=42] Epoch 6/20 Loss: 0.2558  Val AUC: 0.9058
[densenet169 | XR_WRIST | seed=42] Epoch 7/20 Loss: 0.2302  Val AUC: 0.9072
[densenet169 | XR_WRIST | seed=42] Epoch 8/20 Loss: 0.1995  Val AUC: 0.8932
[densenet169 | XR_WRIST | seed=42] Epoch 9/20 Loss: 0.1759  Val AUC: 0.8927
[densenet169 | XR_WRIST | seed=42] Epoch 10/20 Loss: 0.1368  Val AUC: 0.8978
[densenet169 | XR_WRIST | seed=42] Epoch 11/20 Loss: 0.1141  Val AUC: 0.8875
[densenet169 | XR_WRIST | seed=42] Epoch 12/20 Loss: 0.0867  Val AUC: 0.8930
[densenet169 | XR_WRIST | seed=42] Epoch 13/20 Loss: 0.0695  Val AUC: 0.8999
[densene

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

[vit | XR_WRIST | seed=42] Epoch 1/20 Loss: 0.6451  Val AUC: 0.7584
[vit | XR_WRIST | seed=42] Epoch 2/20 Loss: 0.5251  Val AUC: 0.8070
[vit | XR_WRIST | seed=42] Epoch 3/20 Loss: 0.4699  Val AUC: 0.8408
[vit | XR_WRIST | seed=42] Epoch 4/20 Loss: 0.4440  Val AUC: 0.8350
[vit | XR_WRIST | seed=42] Epoch 5/20 Loss: 0.4241  Val AUC: 0.8526
[vit | XR_WRIST | seed=42] Epoch 6/20 Loss: 0.4173  Val AUC: 0.8707
[vit | XR_WRIST | seed=42] Epoch 7/20 Loss: 0.3949  Val AUC: 0.8829
[vit | XR_WRIST | seed=42] Epoch 8/20 Loss: 0.3869  Val AUC: 0.8706
[vit | XR_WRIST | seed=42] Epoch 9/20 Loss: 0.3623  Val AUC: 0.8726
[vit | XR_WRIST | seed=42] Epoch 10/20 Loss: 0.3442  Val AUC: 0.8903
[vit | XR_WRIST | seed=42] Epoch 11/20 Loss: 0.3250  Val AUC: 0.8855
[vit | XR_WRIST | seed=42] Epoch 12/20 Loss: 0.3045  Val AUC: 0.8669
[vit | XR_WRIST | seed=42] Epoch 13/20 Loss: 0.2800  Val AUC: 0.8754
[vit | XR_WRIST | seed=42] Epoch 14/20 Loss: 0.2512  Val AUC: 0.8764
[vit | XR_WRIST | seed=42] Epoch 15/20 Loss